The same seq2seq skills cover other tasks too, such as summarization, paraphrasing, and code translation (Python → JS). Once you’ve learned translation, those come almost for free.

In [1]:
%pip install sentencepiece sacremoses

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr", device_map="auto")

In [42]:
from datasets import load_dataset

raw_datasets = load_dataset("Helsinki-NLP/kde4", lang1='en', lang2='fr', trust_remote_code=True)

In [43]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 210173
    })
})

In [44]:
split_datasets = raw_datasets['train'].train_test_split(train_size=0.9, seed=42)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 189155
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 21018
    })
})

In [45]:
split_datasets['validation'] = split_datasets.pop('test')

In [46]:
split_datasets, split_datasets['train'][0]

(DatasetDict({
     train: Dataset({
         features: ['id', 'translation'],
         num_rows: 189155
     })
     validation: Dataset({
         features: ['id', 'translation'],
         num_rows: 21018
     })
 }),
 {'id': '118328',
  'translation': {'en': 'Show Pathfinder Lander Image',
   'fr': "Afficher l'image de Pathfinder LanderImage/ info menu item (should be translated)"}})

In [47]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
inputs = tokenizer('email', return_tensors='pt').to(model.device)

In [49]:
model

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(59514, 512, padding_idx=59513)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(59514, 512, padding_idx=59513)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,)

In [51]:
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=64, num_beams=4)
print(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
# ['e-mail'] (or similar)

AttributeError: 